# Notebook 6b — Focal-Loss Engineering (E-LightGBM)

The primary experiment. This is where GATE-2, GATE-4, Q6, Q8, Q17, Q20 and
Q23 are answered.

## What changed from R01

| Change | Reason |
|---|---|
| **Arms take the identical feature matrix** | `fit_baseline` fitted 38 raw columns, `fit_focal_lightgbm` fitted all 41. The headline −0.013 measured three changes |
| **Reweighting uses one mechanism for both losses** | `is_unbalance=True` is inert under a custom objective, so the baseline got a weighting correction the proposed arm never received. Both now use `sample_weight` |
| **Full 2×2×2 grid**, including raw + focal | Q20: the cell that isolates the engineered component was never run |
| Repeated CV on the **training pool only** | GATE-1(ii): `rskf.split(X, y)` put the test rows inside CV folds |
| Ten seeds through the same pipeline as the headline | Q6/Q23a: R01's ten-seed run used a different pipeline and different arms, so it was not commensurable with Table 3 |
| Between-seed SD **and** within-seed fold SD | J3 |
| Bootstrap CI computed in committed code | R01's (−0.007, 0.009) appeared in R2 and R4 with no computation anywhere |
| γ × α grid | Q8: a null at one point in a two-parameter space is not a null on the method |
| Instance-level difficulty stratification | Q17: the mechanism is per-instance, all R01 evidence was fold-averaged |
| **No test-set scoring** | GATE-1(iii) |

## The headline contrast

`E_all_ce_noW` → `G_all_focal_noW`. Identical 41-column matrix, identical
weighting, one argument different. Everything else in the grid is reported
with a label saying how many components it varies.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      run_grid, two_level_variance, compare_arms, bootstrap_ci,
                      ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE, fit_arm,
                      score_binary)

banner("NOTEBOOK 6b — E-LIGHTGBM")
OUT = run_dir("notebook06b_focal")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

N_POS = int(train_pool[TARGET].sum())
BASE_RATE = float(train_pool[TARGET].mean())
print(f"train_pool {train_pool.shape}, {N_POS} dropout ({100*BASE_RATE:.1f}%)")
print(f"test_holdout {test_holdout.shape}  -- NOT scored in this notebook")
print(f"\ngamma={GAMMA_REPORTED}, alpha={ALPHA_REPORTED} (config.py)")

print("\nARM GRID")
grid_tbl = pd.DataFrame(ARMS).T
grid_tbl["label"] = grid_tbl.index.map(ARM_LABELS)
print(grid_tbl.to_string())
grid_tbl.to_csv(OUT / "arm_definitions.csv")
print(f"\nheadline  (1 change) : {HEADLINE[0]} -> {HEADLINE[1]}")
print(f"as published (3 changes): {OLD_HEADLINE[0]} -> {OLD_HEADLINE[1]}")

In [ ]:
# ---- verify the is_unbalance claim before relying on it ---------------
# R5's replacement wording asserts is_unbalance is inert under a custom
# objective. Check it rather than asserting it.
from lightgbm import LGBMClassifier
from losses import make_focal, predict_proba_focal

tr, vl = cv_splits(train_pool, SPLIT_SEED)[0]
X_tr, y_tr, X_vl, y_vl, _ = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl])

obj, ev = make_focal(GAMMA_REPORTED, ALPHA_REPORTED)
m_off = LGBMClassifier(objective=obj, is_unbalance=False,
                       random_state=SPLIT_SEED, **SHARED_PARAMS).fit(X_tr, y_tr)
m_on = LGBMClassifier(objective=obj, is_unbalance=True,
                      random_state=SPLIT_SEED, **SHARED_PARAMS).fit(X_tr, y_tr)
p_off = predict_proba_focal(m_off, X_vl)
p_on = predict_proba_focal(m_on, X_vl)
inert = bool(np.allclose(p_off, p_on))

print(f"is_unbalance under a custom objective: "
      f"{'INERT (predictions identical)' if inert else 'ACTIVE (predictions differ)'}")
print(f"  max |difference| = {np.abs(p_off-p_on).max():.2e}")
pd.DataFrame([{"is_unbalance_inert_under_custom_objective": inert,
               "max_abs_prediction_difference": float(np.abs(p_off-p_on).max())}
              ]).to_csv(OUT / "is_unbalance_inertness_check.csv", index=False)
if inert:
    print("\nCONFIRMED. So the R01 baseline received an explicit class-weighting")
    print("correction (is_unbalance=True) that the proposed arm never received,")
    print("beyond the focal alpha term. That is a FOURTH asymmetry on top of the")
    print("three the examination named, and it belongs in the R5 rewrite.")
else:
    print("\nNOT confirmed on this LightGBM version — adjust the R5 wording to match.")

In [ ]:
# ---- the main run: 10 seeds x 25 folds x 9 arms, training pool only ---
print(f"{len(SEEDS)} seeds x {N_SPLITS*N_REPEATS} folds x {len(ARMS)} arms "
      f"= {len(SEEDS)*N_SPLITS*N_REPEATS*len(ARMS)} fits\n")
fold_df, preds = run_grid(train_pool, seeds=SEEDS, collect_predictions=True)
fold_df.to_csv(OUT / "grid_fold_scores.csv", index=False)
preds.to_csv(OUT / "grid_fold_predictions.csv", index=False)
print(f"\n{len(fold_df)} fold-level rows, {len(preds)} per-instance predictions")

In [ ]:
# ---- variance, both levels (Q6, J3) -----------------------------------
var = two_level_variance(fold_df, PRIMARY_METRIC)
var.to_csv(OUT / "variance_two_level.csv", index=False)
print("VARIANCE — between-seed AND within-seed fold SD, same pipeline\n")
print(var.round(4).to_string(index=False))
print(f"\nR01's Table 3 reported '+/- 0.021 across 25 folds'. That is the "
      f"within-seed fold SD column at one seed — five repeats of one "
      f"partitioning scheme, not 25 independent observations.")
print(f"Carry '{N_POS} dropout cases, base rate {100*BASE_RATE:.1f}%' beside "
      "every delta below (J4).")

In [ ]:
# ---- every contrast, labelled by how many components it varies -------
CONTRASTS = [
    ("MATCHED (1 change: objective only)",        *HEADLINE),
    ("AS PUBLISHED (3 changes)",                  *OLD_HEADLINE),
    ("composites only (1 change)",                "A_raw_ce_noW", "E_all_ce_noW"),
    ("focal on raw features (1 change) [MISSING CELL IN R01]",
                                                  "A_raw_ce_noW", "C_raw_focal_noW"),
    ("reweighting only, cross-entropy (1 change)", "E_all_ce_noW", "F_all_ce_W"),
    ("reweighting on top of focal (1 change)",    "G_all_focal_noW", "H_all_focal_W"),
    ("R01 'isolated loss effect' (2 changes)",    "F_all_ce_W", "G_all_focal_noW"),
]
per_seed_all, summaries = [], []
for label, ref, test in CONTRASTS:
    ps, s = compare_arms(fold_df, ref, test, label)
    per_seed_all.append(ps); summaries.append(s)

per_seed_all = pd.concat(per_seed_all, ignore_index=True)
summary_all = pd.DataFrame(summaries)
per_seed_all.to_csv(OUT / "contrast_per_seed.csv", index=False)
summary_all.to_csv(OUT / "contrast_summary.csv", index=False)

print("CONTRAST SUMMARY — every row says how many components it varies\n")
print(summary_all[["contrast", "grand_mean_diff", "between_seed_sd",
                   "ci95_lo", "ci95_hi", "n_seeds_favouring_test",
                   "n_seeds_favouring_ref", "n_seeds_p_below_alpha",
                   "sd_exceeds_effect"]].round(5).to_string(index=False))

print("\n\nTABLE 5 REPLACEMENT — the matched contrast, per seed\n")
m = per_seed_all[per_seed_all["contrast"].str.startswith("MATCHED")]
print(m[["seed", "ref_mean", "test_mean", "diff_mean", "sign",
         "wilcoxon_p", "cohens_d", "ci95_lo", "ci95_hi"]].round(4).to_string(index=False))

h = summary_all[summary_all["contrast"].str.startswith("MATCHED")].iloc[0]
print(f"\nheadline: {h['grand_mean_diff']:+.4f} AUC-PR, between-seed SD "
      f"{h['between_seed_sd']:.4f}, 95% CI [{h['ci95_lo']:+.4f}, {h['ci95_hi']:+.4f}]")
if h["sd_exceeds_effect"]:
    print("The between-seed SD is at least as large as the effect. Clearance "
          "condition C1 names that as a failure; report it as inconclusive.")

In [ ]:
# ---- capacity check (Q23d) --------------------------------------------
cap = (fold_df.groupby("arm")[["pred_std", "pred_min", "pred_max", "pred_mean"]]
       .mean().reset_index())
cap["base_rate"] = BASE_RATE
cap["label"] = cap["arm"].map(ARM_LABELS)
cap.to_csv(OUT / "capacity_check.csv", index=False)
print("Predicted-probability spread per arm (mean over folds)\n")
print(cap.round(4).to_string(index=False))

ce_std = cap[cap["arm"] == HEADLINE[0]]["pred_std"].iloc[0]
fo_std = cap[cap["arm"] == HEADLINE[1]]["pred_std"].iloc[0]
print(f"\nmatched pair: cross-entropy SD {ce_std:.4f} vs focal SD {fo_std:.4f} "
      f"(ratio {fo_std/ce_std:.2f})")
print("A collapsed focal spread is a capacity limit — a publishable diagnosis, "
      "not a failure. A comparable spread keeps playbook causes 3 and 5 ruled "
      "out on evidence, which is where Q23(d) already stood.")

In [ ]:
# ---- gamma x alpha grid (Q8) ------------------------------------------
GAMMA_GRID = [0.0, 1.0, 2.0, 5.0]     # 0.0 reduces focal loss to weighted CE
ALPHA_GRID = [0.50, 0.75, 0.90]
GA_SEEDS = SEEDS[:5]
print(f"{len(GAMMA_GRID)}x{len(ALPHA_GRID)} grid over {len(GA_SEEDS)} seeds")

ga = []
for gm in GAMMA_GRID:
    for al in ALPHA_GRID:
        fd, _ = run_grid(train_pool, seeds=GA_SEEDS, arms=HEADLINE,
                         gamma=gm, alpha=al, verbose=False)
        _, s = compare_arms(fd, HEADLINE[0], HEADLINE[1], f"g{gm}_a{al}")
        ga.append({"gamma": gm, "alpha": al,
                   "diff_mean": s["grand_mean_diff"],
                   "between_seed_sd": s["between_seed_sd"],
                   "ci95_lo": s["ci95_lo"], "ci95_hi": s["ci95_hi"],
                   "n_seeds_favouring_focal": s["n_seeds_favouring_test"],
                   "is_reported_setting": (gm == GAMMA_REPORTED
                                           and al == ALPHA_REPORTED)})
        print(f"  gamma={gm:>3} alpha={al:.2f}: {ga[-1]['diff_mean']:+.4f}")

ga = pd.DataFrame(ga)
ga.to_csv(OUT / "gamma_alpha_grid.csv", index=False)
print("\nFULL GRID — report every cell, including the ones that lose\n")
print(ga.round(4).to_string(index=False))
print("\npivot of the mean difference (focal minus cross-entropy):")
print(ga.pivot(index="gamma", columns="alpha", values="diff_mean").round(4).to_string())

best = ga.loc[ga["diff_mean"].idxmax()]
print(f"\nbest cell: gamma={best['gamma']}, alpha={best['alpha']}, "
      f"{best['diff_mean']:+.4f}")
if (ga["diff_mean"] <= 0).all():
    print("\nNo cell favours focal loss. The claim becomes 'focal loss did not "
          "help anywhere in this region of its parameter space on this data' — "
          "far stronger than R01's single-point null.")

plt.figure(figsize=(6, 4))
piv = ga.pivot(index="gamma", columns="alpha", values="diff_mean")
sns.heatmap(piv, annot=True, fmt=".4f", center=0, cmap="RdBu_r")
plt.title("Focal minus cross-entropy AUC-PR, by gamma and alpha")
plt.tight_layout(); plt.savefig(OUT / "figures/gamma_alpha_heatmap.png", dpi=200)
plt.close()

In [ ]:
# ---- instance-level stratification (Q17, Q23h, Q24) -------------------
# The mechanism reweights per instance; all R01 evidence was fold-averaged.
# Three levels separated the claim from the measurement. Uses predictions
# already collected above -- no new fitting.
ref_arm, foc_arm = HEADLINE
w = preds.pivot_table(index=["seed", "fold", "row_id"], columns="arm",
                      values="p", aggfunc="first")
ylab = (preds.drop_duplicates(["seed", "fold", "row_id"])
        .set_index(["seed", "fold", "row_id"])["y"])
pair = pd.DataFrame({"p_ref": w[ref_arm], "p_focal": w[foc_arm],
                     "y": ylab}).dropna()
pair["delta_p"] = pair["p_focal"] - pair["p_ref"]
pair["boundary_distance"] = (pair["p_ref"] - 0.5).abs()
# difficulty is defined by the REFERENCE arm, never by the arm under test
pair["p_true_class"] = np.where(pair["y"] == 1, pair["p_ref"], 1 - pair["p_ref"])
pair["difficulty"] = pd.cut(pair["p_true_class"], [-.001, .25, .5, .75, 1.0],
                            labels=["very hard", "hard", "moderate", "easy"])
pair["moved_toward_truth"] = np.where(pair["y"] == 1,
                                      pair["delta_p"] > 0, pair["delta_p"] < 0)

strat = (pair.groupby(["difficulty", "y"], observed=True)
         .agg(n=("delta_p", "size"), mean_p_ref=("p_ref", "mean"),
              mean_p_focal=("p_focal", "mean"),
              mean_delta_p=("delta_p", "mean"), sd_delta_p=("delta_p", "std"),
              frac_toward_truth=("moved_toward_truth", "mean"))
         .reset_index())
strat.to_csv(OUT / "instance_level_difficulty.csv", index=False)
pair.reset_index().to_csv(OUT / "instance_level_pairs.csv", index=False)

print("INSTANCE-LEVEL EFFECT BY DIFFICULTY (y=1 is dropout)\n")
print(strat.round(4).to_string(index=False))

hard_pos = pair[(pair["y"] == 1) & (pair["difficulty"].isin(["very hard", "hard"]))]
print(f"\nhard dropout instances: n = {len(hard_pos)}")
if len(hard_pos):
    print(f"  mean change in predicted probability : {hard_pos['delta_p'].mean():+.4f}")
    print(f"  fraction moved toward the true label : {hard_pos['moved_toward_truth'].mean():.3f}")
near = pair[pair["boundary_distance"] < 0.15]
print(f"boundary cases (|p_ref - 0.5| < 0.15): n = {len(near)}, "
      f"mean |delta_p| = {near['delta_p'].abs().mean():.4f}")
print("""
HOW TO READ THIS (it decides the null-diagnosis route):
  * movement near zero, fraction near 0.5 -> the weighting did not move the
    cases it was designed to move. A capacity/granularity finding: playbook
    cause 12, which is where Q24 already routed you.
  * clear movement toward truth on hard cases, flat aggregate metric -> the
    mechanism works and the metric cannot see it. Re-routes to cause 14, and
    it is the more interesting result.
Either is a better Discussion than the aggregate silence R01 reported.""")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=pair.reset_index(), x="difficulty", y="delta_p",
            hue="y", ax=axes[0])
axes[0].axhline(0, c="k", lw=1); axes[0].set_title("Change in predicted probability")
s = pair.sample(min(4000, len(pair)), random_state=SPLIT_SEED)
axes[1].scatter(s["p_ref"], s["p_focal"], s=4, alpha=.3,
                c=np.where(s["y"] == 1, "crimson", "steelblue"))
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("cross-entropy arm"); axes[1].set_ylabel("focal arm")
axes[1].set_title("Per-instance predicted probabilities")
plt.tight_layout(); plt.savefig(OUT / "figures/instance_level.png", dpi=200)
plt.close()

In [ ]:
# ---- persist the fitted headline models for Notebooks 7 and 8 ---------
from pipeline import save_model_bundle, load_model_bundle

X_tr_f, y_tr_f, X_te_f, y_te_f, meta_f = preprocess_inside_fold(
    train_pool, test_holdout)
saved = {}
for arm in set(list(HEADLINE) + list(OLD_HEADLINE)):
    model, predict, cols = fit_arm(arm, X_tr_f, y_tr_f, SPLIT_SEED)
    save_model_bundle(model, cols, arm, MODELS / f"arm_{arm}.pkl")
    saved[arm] = f"arm_{arm}.pkl"
    print(f"saved {arm} -> models/arm_{arm}.pkl ({len(cols)} features)")

# verify the round trip, because this is where R01 hit a wall
for arm in saved:
    pr, cols, payload = load_model_bundle(MODELS / f"arm_{arm}.pkl")
    p_reload = pr(X_tr_f.head(20))
    _, pr_direct, _ = fit_arm(arm, X_tr_f, y_tr_f, SPLIT_SEED)
    print(f"  {arm}: reload matches refit -> "
          f"{np.allclose(p_reload, pr_direct(X_tr_f.head(20)), atol=1e-8)}")

print("""
WHY save_model_bundle() RATHER THAN joblib.dump(model):
LightGBM serialises a custom objective by qualified name. A focal-loss model
pickled directly fails to load unless an identically-named function exists in
the loading namespace -- which is why the R01 Notebook 8 redefined
focal_loss_lgb in its own __main__ just so pickle could resolve the
reference. That workaround is exactly how the three copies of the focal loss
drifted apart. save_model_bundle() stores the trained booster as text and
drops the objective, which was only ever needed during training.""")
print("\nNOTE: these models are fitted on the FULL training pool for "
      "explanation and final scoring. They are NOT scored on the test set "
      "here -- Notebook 8 does that once, after the freeze.")

write_manifest(OUT, {
    "notebook": "06b_focal", "test_set_scored": False,
    "seeds": SEEDS, "arms": list(ARMS), "headline": list(HEADLINE),
    "old_headline": list(OLD_HEADLINE),
    "is_unbalance_inert": inert,
    "n_features_per_fold": int(meta_f["n_features"]),
    "gamma_alpha_cells": int(len(ga)),
    "headline_grand_mean_diff": float(h["grand_mean_diff"]),
    "headline_between_seed_sd": float(h["between_seed_sd"]),
    "saved_models": saved,
})
print("\nNEXT: Notebook 7 (signed SHAP).")